<a href="https://colab.research.google.com/github/Anemodude/Air-quality-index-report-system-/blob/main/M602_Computer_programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import json

class WeatherApi:
    def __init__(self, latitude, longitude):
        self.latitude = latitude
        self.longitude = longitude
        self.base_url = "https://api.open-meteo.com/v1/forecast"

    def fetch_weather(self):
        params = {
            "latitude": self.latitude,
            "longitude": self.longitude,
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation"
        }

        try:
            response = requests.get(self.base_url, params=params)
            response.raise_for_status()
            data = response.json()

            with open("weather_raw.json", "w") as f:
                json.dump(data, f, indent=4)

            return data

        except requests.exceptions.RequestException as e:
            print(f"Weather API Error: {e}")
            return None


In [2]:
import requests
import json

#request
class AirQualityApi:
    def __init__(self, latitude, longitude):
        self.latitude = latitude
        self.longitude = longitude
        self.base_url ="https://air-quality-api.open-meteo.com/v1/air-quality"

    def air_quality_fetch(self):
        params = {
            "latitude": self.latitude,
            "longitude": self.longitude,
            "hourly": ",".join([
                "pm10",
                "pm2_5",
                "carbon_monoxide",
                "nitrogen_dioxide",
                "sulphur_dioxide",
                "ozone",
                "dust",
                "uv_index"
            ])
        }

        try:
            response = requests.get(self.base_url, params=params)
            response.raise_for_status()
            data = response.json()

            with open("air_quality_raw.json", "w") as f:
                json.dump(data, f, indent=4)

            return data

        except requests.exceptions.RequestException as e:
            print(f"API Error: {e}")
            return None


In [3]:
import csv
class AirQualityProcessing:
    def __init__(self, data):
        self.data = data

    def get_condition(self,pm10):
      if pm10 is None:
        return "Unknown"
      if pm10 <0:
        return "Invalid"
      if pm10 <= 20:
        return "Good"
      elif pm10 <= 40:
        return "Fair"
      elif pm10 <= 50:
        return "Moderate"
      elif pm10 <= 100:
        return "Poor"
      elif pm10 <= 150:
        return "Very Poor"
      else:
        return "Extremely Poor"

    def save_csv(self, filename="air_quality_index_report.csv"):
        try:
            hourly = self.data["hourly"]

            date=[]
            time=[]

            for t in hourly["time"]:
                date_part, time_part = t.split("T")
                date.append(date_part)
                time.append(time_part)

            rows=[]

            for i in range(len(hourly["time"])):
                pm10 = hourly["pm10"][i]
                condition = self.get_condition(pm10)

                rows.append([
                    date[i],
                    time[i],
                    pm10,
                    hourly["pm2_5"][i],
                    hourly["carbon_monoxide"][i],
                    hourly["nitrogen_dioxide"][i],
                    hourly["sulphur_dioxide"][i],
                    hourly["ozone"][i],
                    hourly["dust"][i],
                    hourly["uv_index"][i],
                    condition
                ])

            with open(filename, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    "date",
                    "time",
                    "pm10",
                    "pm2_5",
                    "carbon_monoxide",
                    "nitrogen_dioxide",
                    "sulphur_dioxide",
                    "ozone",
                    "dust",
                    "uv_index",
                    "condition"
                ])
                writer.writerows(rows)

        except Exception as e:
            print(f"CSV Error: {e}")

    def correlate_with_weather(self, weather_data):
        try:
            hourly_weather = weather_data["hourly"]

            wind = hourly_weather["wind_speed_10m"]
            humidity = hourly_weather["relative_humidity_2m"]
            temp = hourly_weather["temperature_2m"]
            rain = hourly_weather["precipitation"]

            pm10 = self.data["hourly"]["pm10"]

            correlations = []

            for i in range(len(pm10)):
                explanation = ""

                if pm10[i] > 80 and wind[i] < 5:
                    explanation = "High PM10 likely due to low wind (pollution trapped)."

                elif pm10[i] > 60 and humidity[i] > 80:
                    explanation = "High humidity may be increasing PM10 concentration."

                elif pm10[i] < 30 and rain[i] > 0:
                    explanation = "Rainfall likely reduced PM10 levels."

                elif pm10[i] > 70 and temp[i] < 0:
                    explanation = "Cold air may be trapping pollutants near the ground."

                else:
                    explanation = "No strong weather influence detected."

                correlations.append(explanation)

            return correlations

        except Exception as e:
            print(f"Correlation Error: {e}")
            return []

    def generate_summary(self):
        pm10 = self.data["hourly"]["pm10"]

        summary = {
            "max_pm10": max(pm10),
            "min_pm10": min(pm10),
            "avg_pm10": sum(pm10) / len(pm10),
            "unhealthy_hours": sum(1 for x in pm10 if x > 50),
            "worst_hour_index": pm10.index(max(pm10))
        }

        return summary




In [4]:
class City:
    def __init__(self, filename="city.txt"):
        self.filename = filename

    def add_city(self, city):
        try:
            with open(self.filename, "a") as f:
                f.write(city + "\n")
        except Exception as e:
            print(f"File Error: {e}")

    def load_city(self):
        try:
            with open(self.filename, "r") as f:
                return [line.strip() for line in f.readlines()]
        except FileNotFoundError:
            return []


In [5]:
# City coordinates in our case it was (Potsdam coordinates)
Latitude = 52.3989
Longitude = 13.0657

def main():
    print("Fetching air quality and weather data for Potsdam...")

    api = AirQualityApi(Latitude, Longitude)
    weather_api = WeatherApi(Latitude, Longitude)

    data = api.air_quality_fetch()
    weather = weather_api.fetch_weather()

    if data and weather:
        processor = AirQualityProcessing(data)

        # Save CSV as before
        processor.save_csv()

        # Generate summary
        summary = processor.generate_summary()
        print("\nSummary  of air quality as follows : ")
        print(f"Max PM10: {summary['max_pm10']}")
        max_condition = processor.get_condition(summary['max_pm10'])
        print(f"Condition at Max PM10: {max_condition}")

        print(f"Min PM10: {summary['min_pm10']}")
        print(f"Average PM10: {summary['avg_pm10']:.2f}")
        print(f"Unhealthy Hours: {summary['unhealthy_hours']}")
        print(f"Worst Hour Index: {summary['worst_hour_index']}")

        # Weather correlation
        correlations = processor.correlate_with_weather(weather)
        print("\nWeather Correlation Insights")
        for i, c in enumerate(correlations[:10]):  # show first 10 hours
            print(f"Hour {i}: {c}")

        # Save city
        Air_index = City()
        Air_index.add_city("Potsdam")

        print("\nAir quality + weather report generated successfully.")

if __name__ == "__main__":
    main()


Fetching air quality and weather data for Potsdam...

Summary  of air quality as follows : 
Max PM10: 14.4
Condition at Max PM10: Good
Min PM10: 3.5
Average PM10: 8.40
Unhealthy Hours: 0
Worst Hour Index: 92

--- Weather Correlation Insights ---
Hour 0: No strong weather influence detected.
Hour 1: No strong weather influence detected.
Hour 2: No strong weather influence detected.
Hour 3: No strong weather influence detected.
Hour 4: No strong weather influence detected.
Hour 5: No strong weather influence detected.
Hour 6: No strong weather influence detected.
Hour 7: No strong weather influence detected.
Hour 8: No strong weather influence detected.
Hour 9: Rainfall likely reduced PM10 levels.

Air quality + weather report generated successfully.
